# Board Game Recommender

**Models implemented here:**
1. **Basic**: SVD Matrix Factorization (Funk-SVD with biases, PyTorch)
2. **Advanced**: Neural Collaborative Filtering (NeuMF: GMF + MLP fusion)

**Dataset**: BoardGameGeek Reviews from Kaggle (`jvanelteren/boardgamegeek-reviews`). After filtering (≥5 ratings/user, ≥20 ratings/game), this is ~18.7M ratings on a 1–10 scale across ~272K users and ~21.8K games.

**Evaluation**:
- **RMSE** on the full test set (one held-out rating per user)
- **Recall@10 and NDCG@10** on a 10K-user subsample, with 99 random negatives per positive (standard NCF-style ranking protocol)

## Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Drive mount skipped: {e}")

!pip install -q scikit-learn pandas numpy torch tqdm pyarrow

Mounted at /content/drive


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cpu':
    print("WARNING: Running on CPU. Training might take long")

Device: cuda


## Download Dataset from Kaggle

In [ ]:
# Upload kaggle.json if not already present
if IN_COLAB and not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print("Upload your kaggle.json (Account -> Create New API Token on kaggle.com):")
    files.upload()
    !mkdir -p /root/.kaggle
    !mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

# Download dataset (skip if already downloaded)
if not os.path.exists('boardgamegeek-reviews.zip'):
    !kaggle datasets download -d jvanelteren/boardgamegeek-reviews

if not os.path.exists('bgg_data/bgg-19m-reviews.csv'):
    !unzip -q boardgamegeek-reviews.zip -d bgg_data

!ls -lh bgg_data

Upload your kaggle.json (Account -> Create New API Token on kaggle.com):


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/jvanelteren/boardgamegeek-reviews
License(s): other
100% 1.61G/1.61G [00:17<00:00, 98.3MB/s]

total 5.1G
-rw-r--r-- 1 root root 3.6M Feb  1  2025 2020-08-19.csv
-rw-r--r-- 1 root root 4.9M Feb  1  2025 2022-01-08.csv
-rw-r--r-- 1 root root 1.3G Feb  1  2025 bgg-15m-reviews.csv
-rw-r--r-- 1 root root 1.6G Feb  1  2025 bgg-19m-reviews.csv
-rw-r--r-- 1 root root 2.1G Feb  1  2025 bgg-26m-reviews.csv
-rw-r--r-- 1 root root 105M Feb  1  2025 games_detailed_info2025.csv
-rw-r--r-- 1 root root  95M Feb  1  2025 games_detailed_info.csv


## Build Train/Val/Test Splits

**Filtering**: Keeps users with ≥ 5 ratings and games with ≥ 20 ratings. Removes the long tail of cold users/games where evaluation would be noisy.

**Split**: Per-user **leave-one-out** — for each user, hold 1 random rating for test, 1 for validation, the rest for training.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Load reviews CSV with only relevant columns
DATA_DIR = 'bgg_data'
reviews = pd.read_csv(f'{DATA_DIR}/bgg-19m-reviews.csv',
                      usecols=['user', 'rating', 'ID', 'name'])
reviews = reviews.rename(columns={'ID': 'game_id'})
reviews = reviews.dropna(subset=['user', 'rating', 'game_id'])
reviews['rating'] = reviews['rating'].astype(float)
reviews = reviews[(reviews['rating'] >= 1) & (reviews['rating'] <= 10)]
print(f"Loaded {len(reviews):,} raw reviews")

MIN_USER_RATINGS = 5
MIN_GAME_RATINGS = 20
for _ in range(3):
    ucounts = reviews['user'].value_counts()
    gcounts = reviews['game_id'].value_counts()
    reviews = reviews[
        reviews['user'].isin(ucounts[ucounts >= MIN_USER_RATINGS].index) &
        reviews['game_id'].isin(gcounts[gcounts >= MIN_GAME_RATINGS].index)
    ]
print(f"After filtering: {len(reviews):,} reviews, "
      f"{reviews['user'].nunique():,} users, {reviews['game_id'].nunique():,} games")

# Encode user / game ids to contiguous integers
user_enc = LabelEncoder()
game_enc = LabelEncoder()
reviews['user_idx'] = user_enc.fit_transform(reviews['user'])
reviews['game_idx'] = game_enc.fit_transform(reviews['game_id'])

N_USERS = reviews['user_idx'].nunique()
N_GAMES = reviews['game_idx'].nunique()
print(f"N_USERS={N_USERS}, N_GAMES={N_GAMES}")

Loaded 18,964,728 raw reviews
After filtering: 18,716,519 reviews, 272,319 users, 21,802 games
N_USERS=272319, N_GAMES=21802


In [ ]:
# Per-user leave-one-out split
def leave_one_out_split(df, seed=SEED):
    """For each user: shuffle their ratings, hold out 1 for test, 1 for val, rest train."""
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    df['_rank'] = df.groupby('user_idx').cumcount()
    df['_count'] = df.groupby('user_idx')['user_idx'].transform('count')
    test  = df[df['_rank'] == 0].copy()
    val   = df[df['_rank'] == 1].copy()
    train = df[df['_rank'] >= 2].copy()
    return (train.drop(columns=['_rank', '_count']),
            val.drop(columns=['_rank', '_count']),
            test.drop(columns=['_rank', '_count']))

train_df, val_df, test_df = leave_one_out_split(reviews)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# Free the raw reviews df
del reviews

Train: 18,171,881 | Val: 272,319 | Test: 272,319


## Evaluation Harness

Two metrics:

- **Rating prediction (RMSE)**: full test set
- **Top-K ranking (Recall@10, NDCG@10)**: subsampled 10K users, 99 random negatives per positive — standard NCF-style protocol. Set `N_EVAL_USERS = None` to evaluate on the full test set (slower).

In [ ]:
N_EVAL_USERS = 10000  # subsample for ranking eval; set to None for full test set

def rmse(preds, actuals):
    preds = np.asarray(preds); actuals = np.asarray(actuals)
    return float(np.sqrt(np.mean((preds - actuals) ** 2)))


def _sample_eval_subset(test_df, n=N_EVAL_USERS, seed=SEED):
    if n is None or n >= len(test_df):
        return test_df
    return test_df.sample(n=n, random_state=seed).reset_index(drop=True)


def _build_user_seen_arrays(train_df):
    user_seen = {}
    for u, grp in train_df.groupby('user_idx', sort=False)['game_idx']:
        user_seen[int(u)] = grp.values.astype(np.int32)
    return user_seen


def evaluate_ranking(model_score_fn, test_df, train_df, n_negatives=99, k=10, seed=SEED):
    """Vectorized ranking eval with subsampled test users.

    model_score_fn(user_idx:int, game_idx_array:np.ndarray) -> np.ndarray of scores
    (same length as game_idx_array, higher = better).
    """
    rng = np.random.RandomState(seed)
    eval_df = _sample_eval_subset(test_df)
    user_seen = _build_user_seen_arrays(train_df)

    oversample = int(n_negatives * 1.5) + 5

    recalls, ndcgs = [], []
    users = eval_df['user_idx'].values.astype(np.int64)
    positives = eval_df['game_idx'].values.astype(np.int64)

    for u, pos in tqdm(zip(users, positives), total=len(users), desc="Ranking eval"):
        seen = user_seen.get(int(u), np.array([], dtype=np.int32))
        cand = rng.randint(0, N_GAMES, size=oversample)
        mask = ~np.isin(cand, seen) & (cand != pos)
        cand = cand[mask]
        if len(cand) < n_negatives:
            while len(cand) < n_negatives:
                extra = rng.randint(0, N_GAMES, size=oversample)
                extra = extra[~np.isin(extra, seen) & (extra != pos)]
                cand = np.concatenate([cand, extra])
        negs = cand[:n_negatives]
        candidates = np.concatenate([[pos], negs])
        scores = model_score_fn(int(u), candidates)
        rank = int((scores > scores[0]).sum())
        if rank < k:
            recalls.append(1.0)
            ndcgs.append(1.0 / np.log2(rank + 2))
        else:
            recalls.append(0.0)
            ndcgs.append(0.0)

    return {'recall@10': float(np.mean(recalls)),
            'ndcg@10': float(np.mean(ndcgs)),
            'n_eval': len(users)}

## SVD Matrix Factorization (Funk-SVD)

Classical Funk-SVD implemented in PyTorch for GPU.

**Architecture**:

$$\hat{r}_{ui} = \mu + b_u + b_i + p_u \cdot q_i^T$$

- $\mu$: global mean rating
- $b_u, b_i$: per-user / per-item bias terms
- $p_u, q_i$: 64-dim latent factor vectors

Output is sigmoid-scaled to the rating range [1, 10] to stabilize training. Trained with **MSE loss + L2 regularization** via Adam.

In [ ]:
# Dataset for rating prediction: (user, game, rating) triples
class RatingDataset(Dataset):
    def __init__(self, df):
        self.users   = torch.LongTensor(df['user_idx'].values)
        self.games   = torch.LongTensor(df['game_idx'].values)
        self.ratings = torch.FloatTensor(df['rating'].values)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, i):
        return self.users[i], self.games[i], self.ratings[i]


train_ds = RatingDataset(train_df)
val_ds   = RatingDataset(val_df)

BATCH = 8192
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE.type=='cuda'))
val_loader   = DataLoader(val_ds, batch_size=BATCH*2, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type=='cuda'))

In [ ]:
class SVDModel(nn.Module):
    """Funk-SVD: global mean + user bias + item bias + dot(user_emb, item_emb).

    Output is sigmoid-scaled into [r_min, r_max] to keep predictions in range.
    """
    def __init__(self, n_users, n_games, emb_dim=64,
                 r_min=1.0, r_max=10.0, global_mean=0.0):
        super().__init__()
        self.r_min = r_min
        self.r_max = r_max

        self.user_emb  = nn.Embedding(n_users, emb_dim)
        self.game_emb  = nn.Embedding(n_games, emb_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.game_bias = nn.Embedding(n_games, 1)

        # Small init for embeddings, zero for biases
        nn.init.normal_(self.user_emb.weight,  std=0.01)
        nn.init.normal_(self.game_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.game_bias.weight)

        # Global mean as a non-trainable buffer
        self.register_buffer('global_mean', torch.tensor(float(global_mean)))

    def forward(self, users, games):
        p = self.user_emb(users)              # (B, D)
        q = self.game_emb(games)              # (B, D)
        b_u = self.user_bias(users).squeeze(-1)
        b_g = self.game_bias(games).squeeze(-1)
        dot = (p * q).sum(dim=1)              # (B,)
        # Raw score: global mean + biases + dot product
        raw = self.global_mean + b_u + b_g + dot
        # Sigmoid-scale into [r_min, r_max] for numerical stability
        return self.r_min + (self.r_max - self.r_min) * torch.sigmoid(raw)

In [ ]:
EMB_DIM_SVD = 64
EPOCHS_SVD  = 10
LR_SVD      = 5e-3
WD_SVD      = 1e-5    # L2 regularization (acts on all params, including embeddings)

global_mean = float(train_df['rating'].mean())
print(f"Global mean rating: {global_mean:.3f}")

svd = SVDModel(N_USERS, N_GAMES, emb_dim=EMB_DIM_SVD,
               r_min=1.0, r_max=10.0, global_mean=global_mean).to(DEVICE)
opt = torch.optim.Adam(svd.parameters(), lr=LR_SVD, weight_decay=WD_SVD)
loss_fn = nn.MSELoss()

best_val_rmse = float('inf')
for epoch in range(EPOCHS_SVD):
    # Train pass
    svd.train()
    train_loss, n_seen = 0.0, 0
    pbar = tqdm(train_loader, desc=f"SVD Epoch {epoch+1}/{EPOCHS_SVD}")
    for u, g, r in pbar:
        u, g, r = u.to(DEVICE, non_blocking=True), g.to(DEVICE, non_blocking=True), r.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        pred = svd(u, g)
        loss = loss_fn(pred, r)
        loss.backward()
        opt.step()
        train_loss += loss.item() * u.size(0)
        n_seen += u.size(0)
    train_rmse_epoch = (train_loss / n_seen) ** 0.5

    # Validation pass
    svd.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for u, g, r in val_loader:
            u, g, r = u.to(DEVICE), g.to(DEVICE), r.to(DEVICE)
            pred = svd(u, g)
            val_loss += ((pred - r) ** 2).sum().item()
            n_val   += u.size(0)
    val_rmse = (val_loss / n_val) ** 0.5

    print(f"  Epoch {epoch+1}: train RMSE={train_rmse_epoch:.4f}  val RMSE={val_rmse:.4f}")
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse

print(f"\nBest val RMSE: {best_val_rmse:.4f}")

Global mean rating: 7.049


SVD Epoch 1/10:   0%|          | 0/2219 [00:00<?, ?it/s]

  Epoch 1: train RMSE=2.2671  val RMSE=2.0507


SVD Epoch 2/10:   0%|          | 0/2219 [00:00<?, ?it/s]

  Epoch 2: train RMSE=1.8364  val RMSE=1.8822


...

  Epoch 8: train RMSE=1.6254  val RMSE=1.6008


SVD Epoch 9/10:   0%|          | 0/2219 [00:00<?, ?it/s]

  Epoch 9: train RMSE=1.6188  val RMSE=1.5947


SVD Epoch 10/10:   0%|          | 0/2219 [00:00<?, ?it/s]

  Epoch 10: train RMSE=1.6151  val RMSE=1.5917

Best val RMSE: 1.5917


In [ ]:
class SVDScorer:
    """Wraps the trained SVDModel to match the eval harness signatures."""
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.model.eval()

    def predict_rating(self, user_idx_arr, game_idx_arr):
        """Batched RMSE prediction in [1, 10]."""
        u = torch.as_tensor(np.asarray(user_idx_arr), dtype=torch.long, device=self.device)
        g = torch.as_tensor(np.asarray(game_idx_arr), dtype=torch.long, device=self.device)
        out = np.empty(len(u), dtype=np.float32)
        CHUNK = 65536
        with torch.no_grad():
            for s in range(0, len(u), CHUNK):
                e = min(s + CHUNK, len(u))
                preds = self.model(u[s:e], g[s:e])
                out[s:e] = preds.cpu().numpy()
        return out

    def score(self, user_idx, game_idx_array):
        """Score a single user against a list of candidate games. Higher = better."""
        u = torch.full((len(game_idx_array),), int(user_idx),
                       dtype=torch.long, device=self.device)
        g = torch.as_tensor(np.asarray(game_idx_array), dtype=torch.long, device=self.device)
        with torch.no_grad():
            return self.model(u, g).cpu().numpy()


svd_scorer = SVDScorer(svd, DEVICE)

# Rating RMSE on full test set
preds = svd_scorer.predict_rating(test_df['user_idx'].values, test_df['game_idx'].values)
svd_rmse = rmse(preds, test_df['rating'].values)
print(f"SVD RMSE: {svd_rmse:.4f}")

# Ranking eval on 10K-user subsample
svd_rank = evaluate_ranking(svd_scorer.score, test_df, train_df)
print(f"SVD Recall@10: {svd_rank['recall@10']:.4f}  NDCG@10: {svd_rank['ndcg@10']:.4f}")

SVD RMSE: 1.5888


Ranking eval:   0%|          | 0/10000 [00:00<?, ?it/s]

SVD Recall@10: 0.0055  NDCG@10: 0.0027


## Neural Collaborative Filtering (NeuMF)

NCF generalizes matrix factorization with neural networks. The full **NeuMF** architecture fuses two parallel branches:

- **GMF (Generalized Matrix Factorization)**: element-wise product of user/item embeddings → linear layer
- **MLP**: concatenate user/item embeddings → deep MLP

The two branches each have **their own embeddings** (so GMF embeddings can specialize for linear interactions and MLP embeddings for non-linear ones), and their outputs are concatenated and passed through a final linear layer.

In [ ]:
class NCFModel(nn.Module):
    """NeuMF = GMF (element-wise product) + MLP (concat + deep), fused.

    Architecture:
        GMF branch:  user_id -> GMF_user_emb (D_gmf)
                     item_id -> GMF_item_emb (D_gmf)
                     z_gmf = user_emb * item_emb           (D_gmf)

        MLP branch:  user_id -> MLP_user_emb (D_mlp)
                     item_id -> MLP_item_emb (D_mlp)
                     z_mlp = MLP([user_emb, item_emb])     (mlp_dims[-1])

        Output:      [z_gmf, z_mlp] -> Linear -> sigmoid -> rescale to [r_min, r_max]
    """
    def __init__(self, n_users, n_games,
                 gmf_dim=64, mlp_dim=64, mlp_hidden=(128, 64, 32),
                 dropout=0.1, r_min=1.0, r_max=10.0):
        super().__init__()
        self.r_min = r_min
        self.r_max = r_max

        # GMF branch
        self.gmf_user = nn.Embedding(n_users, gmf_dim)
        self.gmf_item = nn.Embedding(n_games, gmf_dim)

        # MLP branch
        self.mlp_user = nn.Embedding(n_users, mlp_dim)
        self.mlp_item = nn.Embedding(n_games, mlp_dim)

        # Build MLP layers
        layers = []
        in_dim = 2 * mlp_dim
        for h in mlp_hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        self.mlp = nn.Sequential(*layers)

        # Fusion layer: concat(GMF_out, MLP_out) -> single scalar
        self.fusion = nn.Linear(gmf_dim + mlp_hidden[-1], 1)

        # Init
        for emb in [self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item]:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, users, games):
        # GMF branch: element-wise product
        gmf_u = self.gmf_user(users)
        gmf_i = self.gmf_item(games)
        z_gmf = gmf_u * gmf_i                                # (B, gmf_dim)

        # MLP branch: concat embeddings, run through MLP
        mlp_u = self.mlp_user(users)
        mlp_i = self.mlp_item(games)
        z_mlp = self.mlp(torch.cat([mlp_u, mlp_i], dim=1))   # (B, mlp_hidden[-1])

        # Fuse and rescale to [r_min, r_max]
        z = torch.cat([z_gmf, z_mlp], dim=1)
        raw = self.fusion(z).squeeze(-1)
        return self.r_min + (self.r_max - self.r_min) * torch.sigmoid(raw)

In [ ]:
EPOCHS_NCF = 8
LR_NCF     = 1e-3
WD_NCF     = 1e-6

ncf = NCFModel(N_USERS, N_GAMES,
               gmf_dim=64, mlp_dim=64, mlp_hidden=(128, 64, 32),
               dropout=0.1, r_min=1.0, r_max=10.0).to(DEVICE)
opt = torch.optim.Adam(ncf.parameters(), lr=LR_NCF, weight_decay=WD_NCF)
loss_fn = nn.MSELoss()

print(f"NCF parameters: {sum(p.numel() for p in ncf.parameters()):,}")

best_val_rmse = float('inf')
for epoch in range(EPOCHS_NCF):
    # Train pass
    ncf.train()
    train_loss, n_seen = 0.0, 0
    pbar = tqdm(train_loader, desc=f"NCF Epoch {epoch+1}/{EPOCHS_NCF}")
    for u, g, r in pbar:
        u, g, r = u.to(DEVICE, non_blocking=True), g.to(DEVICE, non_blocking=True), r.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        pred = ncf(u, g)
        loss = loss_fn(pred, r)
        loss.backward()
        opt.step()
        train_loss += loss.item() * u.size(0)
        n_seen += u.size(0)
    train_rmse_epoch = (train_loss / n_seen) ** 0.5

    # Validation pass
    ncf.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for u, g, r in val_loader:
            u, g, r = u.to(DEVICE), g.to(DEVICE), r.to(DEVICE)
            pred = ncf(u, g)
            val_loss += ((pred - r) ** 2).sum().item()
            n_val   += u.size(0)
    val_rmse = (val_loss / n_val) ** 0.5

    print(f"  Epoch {epoch+1}: train RMSE={train_rmse_epoch:.4f}  val RMSE={val_rmse:.4f}")
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse

print(f"\nBest val RMSE: {best_val_rmse:.4f}")

NCF parameters: 37,674,433


NCF Epoch 1/8:   0%|          | 0/2219 [00:00<?, ?it/s]

  Epoch 1: train RMSE=1.2615  val RMSE=1.2679


NCF Epoch 2/8:   0%|          | 0/2219 [00:00<?, ?it/s]

  Epoch 2: train RMSE=1.1265  val RMSE=1.2515


...

  Epoch 6: train RMSE=0.8819  val RMSE=1.3313


NCF Epoch 7/8:   0%|          | 0/2219 [00:00<?, ?it/s]

  Epoch 7: train RMSE=0.8651  val RMSE=1.3392


NCF Epoch 8/8:   0%|          | 0/2219 [00:00<?, ?it/s]

  Epoch 8: train RMSE=0.8520  val RMSE=1.3510

Best val RMSE: 1.2515


In [ ]:
class NCFScorer:
    """Same interface as SVDScorer but for the NCF model."""
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.model.eval()

    def predict_rating(self, user_idx_arr, game_idx_arr):
        u = torch.as_tensor(np.asarray(user_idx_arr), dtype=torch.long, device=self.device)
        g = torch.as_tensor(np.asarray(game_idx_arr), dtype=torch.long, device=self.device)
        out = np.empty(len(u), dtype=np.float32)
        CHUNK = 32768
        with torch.no_grad():
            for s in range(0, len(u), CHUNK):
                e = min(s + CHUNK, len(u))
                preds = self.model(u[s:e], g[s:e])
                out[s:e] = preds.cpu().numpy()
        return out

    def score(self, user_idx, game_idx_array):
        u = torch.full((len(game_idx_array),), int(user_idx),
                       dtype=torch.long, device=self.device)
        g = torch.as_tensor(np.asarray(game_idx_array), dtype=torch.long, device=self.device)
        with torch.no_grad():
            return self.model(u, g).cpu().numpy()


ncf_scorer = NCFScorer(ncf, DEVICE)

# Rating RMSE
preds = ncf_scorer.predict_rating(test_df['user_idx'].values, test_df['game_idx'].values)
ncf_rmse = rmse(preds, test_df['rating'].values)
print(f"NCF RMSE: {ncf_rmse:.4f}")

# Ranking
ncf_rank = evaluate_ranking(ncf_scorer.score, test_df, train_df)
print(f"NCF Recall@10: {ncf_rank['recall@10']:.4f}  NDCG@10: {ncf_rank['ndcg@10']:.4f}")

NCF RMSE: 1.3512


Ranking eval:   0%|          | 0/10000 [00:00<?, ?it/s]

NCF Recall@10: 0.3412  NDCG@10: 0.1797


## Results Summary

In [ ]:
results = pd.DataFrame([
    {'Model': 'SVD (Funk-SVD MF)',  'Type': 'Basic',    'RMSE': svd_rmse, 'Recall@10': svd_rank['recall@10'], 'NDCG@10': svd_rank['ndcg@10']},
    {'Model': 'NCF (NeuMF, MSE)',   'Type': 'Advanced', 'RMSE': ncf_rmse, 'Recall@10': ncf_rank['recall@10'], 'NDCG@10': ncf_rank['ndcg@10']},
])
print(results.to_string(index=False))

            Model     Type     RMSE  Recall@10  NDCG@10
SVD (Funk-SVD MF)    Basic 1.588849     0.0055 0.002748
 NCF (NeuMF, MSE) Advanced 1.351216     0.3412 0.179675


## Save Model Artifacts

Save trained weights, splits, and results to Drive for reuse.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/bgg_phase1_brandon' if IN_COLAB and os.path.exists('/content/drive/MyDrive') else './bgg_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Save model state_dicts
torch.save(svd.state_dict(), f'{OUT_DIR}/svd_brandon.pt')
torch.save(ncf.state_dict(), f'{OUT_DIR}/ncf_brandon.pt')

# Save splits and encoders for later reuse
train_df.to_parquet(f'{OUT_DIR}/train.parquet', index=False)
val_df.to_parquet(f'{OUT_DIR}/val.parquet',     index=False)
test_df.to_parquet(f'{OUT_DIR}/test.parquet',   index=False)
with open(f'{OUT_DIR}/encoders.pkl', 'wb') as f:
    pickle.dump({'user_enc': user_enc, 'game_enc': game_enc,
                 'N_USERS': N_USERS, 'N_GAMES': N_GAMES}, f)

# Save results table
results.to_csv(f'{OUT_DIR}/results_brandon.csv', index=False)

print(f"Saved to {OUT_DIR}:")
print(f"  - svd_brandon.pt  ({EMB_DIM_SVD}-dim embeddings)")
print(f"  - ncf_brandon.pt  (gmf=64, mlp=64, hidden=128/64/32)")
print(f"  - train/val/test.parquet + encoders.pkl")
print(f"  - results_brandon.csv")

Saved to /content/drive/MyDrive/bgg_phase1_brandon:
  - svd_brandon.pt  (64-dim embeddings)
  - ncf_brandon.pt  (gmf=64, mlp=64, hidden=128/64/32)
  - train/val/test.parquet + encoders.pkl
  - results_brandon.csv
